# SHIFT-SURV — rebuilding the paper figures

Every figure in the paper is produced by a script in `paper_assets/` that reads
result tables from `outputs/tables/`. Those tables are tracked in this repository,
so this notebook runs end to end without rerunning any experiment.

Run the cells in order. Each one runs one generator, reports which files it wrote,
and displays them. A generator whose input tables are missing reports what it
needs and the notebook continues.

To regenerate the tables themselves, see the pipeline section of `README.md`.

In [ ]:
import subprocess, sys, time
from pathlib import Path
from IPython.display import Image, display, Markdown

ROOT = Path.cwd()
while not (ROOT / 'paper_assets').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
TABLES = ROOT / 'outputs' / 'tables'
FIGS   = ROOT / 'outputs' / 'figures' / 'paper'
FIGS.mkdir(parents=True, exist_ok=True)
print('repository :', ROOT)
print('tabelle    :', len(list(TABLES.glob('*.csv'))), 'file CSV')
print('python     :', sys.version.split()[0])

## Helper

`run(script)` executes one generator and shows the figures it wrote. Newly written
files are detected by modification time, so the notebook stays correct when a
generator is renamed or starts producing an extra panel.

In [ ]:
def run(script, args=(), show=True, width=560):
    """Run one figure generator and display whatever it wrote."""
    t0 = time.time() - 1
    cmd = [sys.executable, str(ROOT / 'paper_assets' / script), *map(str, args)]
    p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    if p.returncode != 0:
        tail = (p.stderr or p.stdout).strip().splitlines()[-6:]
        display(Markdown(f'**{script} non ha completato.** Ultime righe:'))
        print('\n'.join(tail))
        return []
    if p.stdout.strip():
        print(p.stdout.strip())
    fresh = sorted(f for f in FIGS.glob('*.png') if f.stat().st_mtime >= t0)
    if not fresh:
        display(Markdown(f'_{script}: nessuna figura nuova._'))
    for f in fresh:
        display(Markdown(f'**{f.name}**'))
        if show:
            display(Image(filename=str(f), width=width))
    return fresh

## 1. Synthetic benchmark

Discrimination and calibration per shift regime, under the adaptation protocol
(one source centre) and the domain-generalization protocol (three source centres).

Inputs: `benchmark_1s1t_iwcv_*`, `benchmark_1s1t_prior_summary`,
`benchmark_3s1t_*`, `benchmark_3s1t_prior_summary`.

In [ ]:
run('_gen_fig_disc.py')

In [ ]:
run('_gen_fig_cal.py')

## 2. Real cohorts

Pooled effects on the I3LUNG transfers and on the Rotterdam–GBSG2 registry.

Inputs: `i3lung_meta.csv`, `i3lung_loco_meta.csv`, `public_rotterdam_gbsg_meta.csv`.

In [ ]:
run('_gen_fig_realmain.py')

In [ ]:
run('_gen_fig_realdata_panels.py')

## 3. What a deployment can measure

Covariate distance between two centres against the discrimination ERM loses
going from one to the other. Inputs: `i3lung_*_shiftdiag.csv`, `i3lung_*_summary.csv`.

In [ ]:
run('_gen_fig_predictability.py')

## 4. Remaining figures

Shift diagnostics on I3LUNG, the sample-size sweep, and the failure decomposition.
Some of these appear only in the appendix.

In [ ]:
for s in ('_gen_fig_i3lung_shift.py', '_gen_fig_nsweep.py',
          '_gen_fig_whyfails.py', '_gen_fig3_dissociation.py',
          '_gen_fig_real.py'):
    display(Markdown(f'### {s}'))
    run(s, show=False)

## Inventory

Everything currently in `outputs/figures/paper/`, with the time it was written.

In [ ]:
rows = sorted(FIGS.glob('*.pdf'))
print(f"{len(rows)} figure\n")
for f in rows:
    age = (time.time() - f.stat().st_mtime) / 3600
    print(f'  {f.name:<34} {f.stat().st_size/1024:6.0f} KB   scritta {age:5.1f} h fa')